# 03 — Modélisation, GridSearchCV & tracking MLflow

Ce notebook couvre **Milestone 1 & 4**.

- Setup MLflow
- Pipelines anti data leakage (preprocessing + SMOTE + modèle)
- GridSearchCV sur **score métier**
- Logging des résultats et artefacts (ROC, confusion matrix, modèle)

In [1]:
from pathlib import Path
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score

import sys, os
sys.path.insert(0, os.path.abspath(".."))  # monte d'un niveau (../)
from src.config import PATHS
from src.data import load_parquet
from src.metrics import optimal_threshold_cost, get_proba
from src.models import make_cv, gridsearch_dummy, gridsearch_lgbm_smote, gridsearch_xgb_smote, gridsearch_logreg_smote, gridsearch_rf_smote, run_gridsearch

from src.mlflow_utils import init_mlflow, track_run
import mlflow

## 1) Charger dataset préparé

In [2]:
df = load_parquet(PATHS.data_processed / "train_fe.parquet")

assert "TARGET" in df.columns, "Le dataset doit contenir TARGET (uniquement train)."
X = df.drop(columns=["TARGET"])
y = df["TARGET"].astype(int)

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_valid.shape)
print(set(X.dtypes))
print(X.select_dtypes(include='object').columns.tolist())



(79999, 125) (20000, 125)
{dtype('O'), dtype('bool'), dtype('float64'), dtype('int64')}
['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']


## 2) Init MLflow

In [3]:
from datetime import datetime

init_mlflow()

mlflow.set_experiment("credit_scoring")
notebook_run_id = datetime.now().strftime("%Y%m%d_%H%M%S")


C:\apps\anaconda3\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:177: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/05 18:12:29 INFO mlflow.tracking.fluent: Experiment with name 'credit_scoring' does not exist. Creating a new experiment.


In [4]:
cv = make_cv(2)

In [5]:
from src.models import build_preprocessor

print(X.shape, set(X.dtypes))

pre = build_preprocessor(X)
Xt = pre.fit_transform(X)
print(Xt.shape, Xt.dtype)
print(type(Xt))

(99999, 125) {dtype('O'), dtype('bool'), dtype('float64'), dtype('int64')}
(99999, 256) float64
<class 'numpy.ndarray'>


## 3) LightGBM

In [6]:
pipe, param_grid = gridsearch_lgbm_smote(X_train, y_train, cv)
t0 = time.time()
gs = run_gridsearch(pipe, param_grid, X_train, y_train, cv)
fit_time = time.time() - t0

best_model_lightGBM = gs.best_estimator_
t1 = time.time()
y_proba = get_proba(best_model_lightGBM, X_valid)
predict_time = time.time() - t1

threshold_info = optimal_threshold_cost(y_valid.values, y_proba)
y_pred = (y_proba >= threshold_info["threshold"]).astype(int)

cv_results_df = pd.DataFrame(gs.cv_results_)

extra_metrics_lightGBM = {
    "business_cost_test": threshold_info["cost"],
    "business_threshold_test": threshold_info["threshold"],
    "business_score_cv": gs.best_score_,
    "AUC_test": roc_auc_score(y_valid, y_proba),
    "mean_cv_threshold": gs.cv_results_["mean_test_threshold"][gs.best_index_],
}


Fitting 2 folds for each of 8 candidates, totalling 16 fits
[LightGBM] [Info] Number of positive: 6474, number of negative: 73525
[LightGBM] [Debug] Dataset::GetMultiBinFromSparseFeatures: sparse rate 0.892046
[LightGBM] [Debug] Dataset::GetMultiBinFromAllFeatures: sparse rate 0.471195
[LightGBM] [Debug] init for col-wise cost 0.130612 seconds, init for row-wise cost 0.445720 seconds
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.169543 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Debug] Using Sparse Multi-Val Bin
[LightGBM] [Info] Total Bins 12203
[LightGBM] [Info] Number of data points in the train set: 79999, number of used features: 231
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080926 -> initscore=-2.429831
[LightGBM] [Info] Start training from score -2.429831
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 8
[LightGBM] [D

C:\apps\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


### Résultat

In [7]:

gs.best_params_, gs.best_score_, extra_metrics_lightGBM

({'model__colsample_bytree': 0.8,
  'model__learning_rate': 0.05,
  'model__max_depth': 8,
  'model__min_child_samples': 60,
  'model__n_estimators': 400,
  'model__num_leaves': 31,
  'model__subsample': 0.8},
 np.float64(-21760.0),
 {'business_cost_test': 10365.0,
  'business_threshold_test': 0.09,
  'business_score_cv': np.float64(-21760.0),
  'AUC_test': 0.758976383453669,
  'mean_cv_threshold': np.float64(0.07250000000000001)})

### MLFlow tracking

In [8]:
run_name = "lightgbm"

model_type = f"{run_name}_threshold_cost_optimized"

mlflow.set_tag("algo", run_name)
mlflow.set_tag("model_type", model_type)
mlflow.set_tag("stage", "validation")
mlflow.set_tag("threshold_strategy", "cost_optimized")

mlflow.log_metric("auc", float(roc_auc_score(y_valid, y_valid_proba)))
mlflow.log_metric("f1", float(f1_score(y_valid, y_valid_pred)))
mlflow.log_metric("business_cost", float(threshold_info["cost"]))
mlflow.log_metric("threshold", float(threshold_info["threshold"]))
mlflow.log_metric("fit_time", float(fit_time))
mlflow.log_metric("predict_time", float(predict_time))

mlflow.log_metric("main_score", -float(threshold_info["cost"]))

track_run(
    run_name= run_name,
    model=best_model_lightGBM,
    X_train=X_train, y_train=y_train,
    X_valid=X_valid, y_valid=y_valid,
    params=gs.best_params_,
    extra_metrics=extra_metrics_lightGBM,
    y_valid_proba=y_proba,
    y_valid_pred=y_pred,
    threshold_info=threshold_info,
    fit_time=fit_time,
    predict_time=predict_time,
    cv_results_df=cv_results_df,
    notebook_run_id=notebook_run_id,
)

NameError: name 'y_valid_proba' is not defined

### SAVE model to .joblib

In [ ]:
import joblib

best_model = best_model_lightGBM #gs.best_estimator_

artifact = {
    "model": best_model,
    "threshold": threshold_info["threshold"],
    "threshold_info": threshold_info,
    "score_name": "business_cost_opt",
}

# joblib.dump(best_model, "../model/model.joblib")
joblib.dump(artifact, "../model/best_model_lightGBM.joblib")

print("✅ Model saved to model.joblib")
print("Best params:", gs.best_params_)
print("Best score:", gs.best_score_)

## 4) Baseline DummyClassifier (référence)

In [ ]:
pipe, param_grid = gridsearch_dummy(X_train, y_train, cv)
t0 = time.time()
gs = run_gridsearch(pipe, param_grid, X_train, y_train, cv)
fit_time = time.time() - t0

best_model_dummyClassif = gs.best_estimator_
t1 = time.time()
y_proba = get_proba(best_model_dummyClassif, X_valid)
predict_time = time.time() - t1

threshold_info = optimal_threshold_cost(y_valid.values, y_proba)
y_pred = (y_proba >= threshold_info["threshold"]).astype(int)

# CV results artifact (utile pour stabilité & soutenance)
cv_results_df = pd.DataFrame(gs.cv_results_)

extra_metrics_dummyClassif = {
    "business_cost_test": threshold_info["cost"],
    "business_threshold_test": threshold_info["threshold"],
    "business_score_cv": gs.best_score_,
    "AUC_test": roc_auc_score(y_valid, y_proba),
}


### Résultat

In [ ]:
gs.best_params_, gs.best_score_, extra_metrics_dummyClassif

### MLFlow tracking

In [ ]:
run_name = "dummy_baseline"

model_type = f"{run_name}_threshold_cost_optimized"

mlflow.set_tag("algo", run_name)
mlflow.set_tag("model_type", model_type)
mlflow.set_tag("stage", "validation")
mlflow.set_tag("threshold_strategy", "cost_optimized")

mlflow.log_metric("auc", float(roc_auc_score(y_valid, y_valid_proba)))
mlflow.log_metric("f1", float(f1_score(y_valid, y_valid_pred)))
mlflow.log_metric("business_cost", float(threshold_info["cost"]))
mlflow.log_metric("threshold", float(threshold_info["threshold"]))
mlflow.log_metric("fit_time", float(fit_time))
mlflow.log_metric("predict_time", float(predict_time))

mlflow.log_metric("main_score", -float(threshold_info["cost"]))

track_run(
    run_name=run_name,
    model=best_model_dummyClassif,
    X_train=X_train, y_train=y_train,
    X_valid=X_valid, y_valid=y_valid,
    params=gs.best_params_,
    extra_metrics=extra_metrics_dummyClassif,
    y_valid_proba=y_proba,
    y_valid_pred=y_pred,
    threshold_info=threshold_info,
    fit_time=fit_time,
    predict_time=predict_time,
    cv_results_df=cv_results_df,
    notebook_run_id=notebook_run_id,
)

## 5) XGBoost 

In [ ]:
pipe, param_grid = gridsearch_xgb_smote(X_train, y_train, cv)
t0 = time.time()
gs = run_gridsearch(pipe, param_grid, X_train, y_train, cv)
fit_time = time.time() - t0

best_model_xgBoost = gs.best_estimator_
t1 = time.time()
y_proba = get_proba(best_model_xgBoost, X_valid)
predict_time = time.time() - t1

threshold_info = optimal_threshold_cost(y_valid.values, y_proba)
y_pred = (y_proba >= threshold_info["threshold"]).astype(int)

cv_results_df = pd.DataFrame(gs.cv_results_)

extra_metrics_xgBoost = {
    "business_cost_test": threshold_info["cost"],
    "business_threshold_test": threshold_info["threshold"],
    "business_score_cv": gs.best_score_,
    "AUC_test": roc_auc_score(y_valid, y_proba),
    "mean_cv_threshold": gs.cv_results_["mean_test_threshold"][gs.best_index_],
}

### Résultat

In [ ]:
gs.best_params_, gs.best_score_, extra_metrics_xgBoost

### MLFlow tracking

In [ ]:
run_name = "xgboost"

model_type = f"{run_name}_threshold_cost_optimized"

mlflow.set_tag("algo", run_name)
mlflow.set_tag("model_type", model_type)
mlflow.set_tag("stage", "validation")
mlflow.set_tag("threshold_strategy", "cost_optimized")

mlflow.log_metric("auc", float(roc_auc_score(y_valid, y_valid_proba)))
mlflow.log_metric("f1", float(f1_score(y_valid, y_valid_pred)))
mlflow.log_metric("business_cost", float(threshold_info["cost"]))
mlflow.log_metric("threshold", float(threshold_info["threshold"]))
mlflow.log_metric("fit_time", float(fit_time))
mlflow.log_metric("predict_time", float(predict_time))

mlflow.log_metric("main_score", -float(threshold_info["cost"]))

track_run(
    run_name=run_name,
    model=best_model_xgBoost,
    X_train=X_train, y_train=y_train,
    X_valid=X_valid, y_valid=y_valid,
    params=gs.best_params_,
    extra_metrics=extra_metrics_xgBoost,
    y_valid_proba=y_proba,
    y_valid_pred=y_pred,
    threshold_info=threshold_info,
    fit_time=fit_time,
    predict_time=predict_time,
    cv_results_df=cv_results_df,
    notebook_run_id=notebook_run_id,
)


### SAVE model to .joblib

In [ ]:
import joblib

best_model = best_model_xgBoost #gs.best_estimator_

artifact = {
    "model": best_model,
    "threshold": threshold_info["threshold"],
    "threshold_info": threshold_info,
    "score_name": "business_cost_opt",
}

# joblib.dump(best_model, "../model/model.joblib")
joblib.dump(artifact, "../model/best_model_xgBoost.joblib")

print("✅ Model saved to model.joblib")
print("Best params:", gs.best_params_)
print("Best score:", gs.best_score_)

In [ ]:
# !pip freeze > requirements.txt

In [ ]:
import sklearn
print(sklearn.__version__)

In [ ]:
import imblearn
print(imblearn.__version__)

## 6) Logistic Regression + SMOTE

In [ ]:
pipe, param_grid = gridsearch_logreg_smote(X_train, y_train, cv)
t0 = time.time()
gs = run_gridsearch(pipe, param_grid, X_train, y_train, cv)
fit_time = time.time() - t0

best_model_logRegression = gs.best_estimator_
t1 = time.time()
y_proba = get_proba(best_model_logRegression, X_valid)
predict_time = time.time() - t1

threshold_info = optimal_threshold_cost(y_valid.values, y_proba)
y_pred = (y_proba >= threshold_info["threshold"]).astype(int)

cv_results_df = pd.DataFrame(gs.cv_results_)

extra_metrics_logRegression = {
    "business_cost_test": threshold_info["cost"],
    "business_threshold_test": threshold_info["threshold"],
    "business_score_cv": gs.best_score_,
    "AUC_test": roc_auc_score(y_valid, y_proba),
    "mean_cv_threshold": gs.cv_results_["mean_test_threshold"][gs.best_index_],
}

### Résultats

In [ ]:
gs.best_params_, gs.best_score_, extra_metrics_logRegression

### MLFlow tracking

In [ ]:
run_name = "logreg_smote"

model_type = f"{run_name}_threshold_cost_optimized"

mlflow.set_tag("algo", run_name)
mlflow.set_tag("model_type", model_type)
mlflow.set_tag("stage", "validation")
mlflow.set_tag("threshold_strategy", "cost_optimized")

mlflow.log_metric("auc", float(roc_auc_score(y_valid, y_valid_proba)))
mlflow.log_metric("f1", float(f1_score(y_valid, y_valid_pred)))
mlflow.log_metric("business_cost", float(threshold_info["cost"]))
mlflow.log_metric("threshold", float(threshold_info["threshold"]))
mlflow.log_metric("fit_time", float(fit_time))
mlflow.log_metric("predict_time", float(predict_time))

mlflow.log_metric("main_score", -float(threshold_info["cost"]))

track_run(
    run_name=run_name,
    model=best_model_logRegression,
    X_train=X_train, y_train=y_train,
    X_valid=X_valid, y_valid=y_valid,
    params=gs.best_params_,
    extra_metrics=extra_metrics_logRegression,
    y_valid_proba=y_proba,
    y_valid_pred=y_pred,
    threshold_info=threshold_info,
    fit_time=fit_time,
    predict_time=predict_time,
    cv_results_df=cv_results_df,
    notebook_run_id=notebook_run_id,
)

In [ ]:
import mlflow

print("tracking uri =", mlflow.get_tracking_uri())

exp = mlflow.get_experiment_by_name("credit_scoring")
print("credit_scoring experiment =", exp)

runs = mlflow.search_runs()
print("Nb runs =", len(runs))
print(runs[["run_id", "experiment_id", "tags.mlflow.runName"]].tail(20))

## 7) RandomForest + SMOTE

In [ ]:
pipe, param_grid = gridsearch_rf_smote(X_train, y_train, cv)
t0 = time.time()
gs = run_gridsearch(pipe, param_grid, X_train, y_train, cv)
fit_time = time.time() - t0

best_model_randomForest = gs.best_estimator_
t1 = time.time()
y_proba = get_proba(best_model_randomForest, X_valid)
predict_time = time.time() - t1

threshold_info = optimal_threshold_cost(y_valid.values, y_proba)
y_pred = (y_proba >= threshold_info["threshold"]).astype(int)

cv_results_df = pd.DataFrame(gs.cv_results_)

extra_metrics_randomForest = {
    "business_cost_test": threshold_info["cost"],
    "business_threshold_test": threshold_info["threshold"],
    "business_score_cv": gs.best_score_,
    "AUC_test": roc_auc_score(y_valid, y_proba),
    "mean_cv_threshold": gs.cv_results_["mean_test_threshold"][gs.best_index_],
}

### Résultats

In [ ]:
gs.best_params_, gs.best_score_, extra_metrics_randomForest

### MLFlow tracking

In [ ]:
run_name = "rf_smote"

model_type = f"{run_name}_threshold_cost_optimized"

mlflow.set_tag("algo", run_name)
mlflow.set_tag("model_type", model_type)
mlflow.set_tag("stage", "validation")
mlflow.set_tag("threshold_strategy", "cost_optimized")

mlflow.log_metric("auc", float(roc_auc_score(y_valid, y_valid_proba)))
mlflow.log_metric("f1", float(f1_score(y_valid, y_valid_pred)))
mlflow.log_metric("business_cost", float(threshold_info["cost"]))
mlflow.log_metric("threshold", float(threshold_info["threshold"]))
mlflow.log_metric("fit_time", float(fit_time))
mlflow.log_metric("predict_time", float(predict_time))

mlflow.log_metric("main_score", -float(threshold_info["cost"]))

track_run(
    run_name=run_name,
    model=best_model_randomForest,
    X_train=X_train, y_train=y_train,
    X_valid=X_valid, y_valid=y_valid,
    params=gs.best_params_,
    extra_metrics=extra_metrics_randomForest,
    y_valid_proba=y_proba,
    y_valid_pred=y_pred,
    threshold_info=threshold_info,
    fit_time=fit_time,
    predict_time=predict_time,
    cv_results_df=cv_results_df,
    notebook_run_id=notebook_run_id,
)

## 8) À faire

- Ajouter d'autres algos (GradientBoosting, XGBoost/LightGBM si autorisé)
- Faire un **fine-tuning** autour des meilleurs hyperparamètres
- Comparer via MLflow UI (runs)

# Dana Tests

## Étape 1 — Comparer les algorithmes (simple)
### 1 algorithme = 1 run MLflow

In [ ]:
import mlflow
models = {
    "LogisticRegression": best_model_logRegression,
    "RandomForest": best_model_randomForest,
}

for name, model in models.items():
    model.fit(X_train, y_train)
    auc = roc_auc_score(y_valid, model.predict_proba(X_valid)[:, 1])

    with mlflow.start_run(run_name=name):
        mlflow.log_metric("auc", auc)
        mlflow.log_param("model", name)


## Étape 2 — Affiner le meilleur (toujours simple)
### Supposons que RandomForest gagne.
### 1 run = 1 réglage

In [ ]:
from sklearn.ensemble import RandomForestClassifier
depths = [5, 10, 15]

for d in depths:
    rf = RandomForestClassifier(max_depth=d)
    rf.fit(X_train, y_train)
    auc = roc_auc_score(y_valid, rf.predict_proba(X_valid)[:, 1])

    with mlflow.start_run(run_name=f"RF_depth_{d}"):
        mlflow.log_metric("auc", auc)
        mlflow.log_param("max_depth", d)
